In [ ]:
'''libraries'''
#Data
import pandas as pd

#Plots
import matplotlib.pyplot as plt

#math
import numpy as np

#Constants
from scipy.constants import physical_constants
m_u=physical_constants['atomic mass constant energy equivalent in MeV'][0]
from scipy.constants import speed_of_light as c

#Usefull
from tqdm.notebook import tqdm
import os
from scipy.interpolate import interp1d
import pynucastro as pyna
#%matplotlib widget

Read Data from the Snaps


In [ ]:
folder = 'Example_NSM_dyn_ejecta_rosswog_500dias_sin_theoretical_beta_alpha_out\snaps'

files = os.listdir(folder)

abundances_winnet=[]
time_Winnet=[]
temperature_winnet=[]
density_winnet=[]

for file in tqdm(files):

    abundances_winnet.append(np.loadtxt(os.path.join(folder, file), skiprows=3))
    time_Winnet.append(float(np.loadtxt(os.path.join(folder, file), skiprows=1,max_rows=1, usecols=0)))
    temperature_winnet.append(float(np.loadtxt(os.path.join(folder, file), skiprows=1,max_rows=1, usecols=1)))
    density_winnet.append(float(np.loadtxt(os.path.join(folder, file), skiprows=1,max_rows=1, usecols=2)))

abundances_winnet = np.array(abundances_winnet)
time_Winnet = np.array(time_Winnet) 
density_winnet = np.array(density_winnet)

Interpolation to have a smooth Xi curve

In [ ]:
initial_time=1 #seconds
final_time=4.32E7 #seconds 500 days= 4.32E7 seconds 100 days= 8.64E7 seconds
N_steps=100000
time = np.exp(np.linspace(np.log(initial_time), np.log(final_time), N_steps))
abundances_time= np.empty((abundances_winnet.shape[1],N_steps))
for i in tqdm(range(abundances_winnet.shape[1])):
    spline_interp = interp1d(time_Winnet, abundances_winnet[:,i,3], kind='linear')
    abundances_time[i] = spline_interp(time)
spline_interp_temp=interp1d(time_Winnet,temperature_winnet,kind='linear')
spline_interp_density=interp1d(time_Winnet,density_winnet,kind='linear')
temperature=spline_interp_temp(time)
density=spline_interp_density(time)

def index_out_win_n_z(n,z):
    idx_start = np.searchsorted(abundances_winnet[0,:,1], z, side='left')
    idx_end = np.searchsorted(abundances_winnet[0,:,1], z, side='right')
    if idx_end>idx_start:
        j=np.searchsorted(abundances_winnet[0,:,0][idx_start:idx_end],n)+idx_start
        if abundances_winnet[0,:,0][j]==n and abundances_winnet[0,:,1][j]==z:
            return j
        else:
            return 'None'
    else:
        return 'None'
    
def abundance_time_of_nuclei_n_z(n,z):

    j=index_out_win_n_z(n,z)
    if j=='None':
        return np.zeros(N_steps)
    elif abundances_winnet[0,:,0][j]==n and abundances_winnet[0,:,1][j]==z:
        return abundances_time[j]
    else:
        return np.zeros(N_steps)
    

Properties of nuclei from Winvne_v2.0

In [ ]:
nuclear_data=pd.read_csv('Nuclear_data\Mass\winvne_v2.0.dat',skiprows=lambda x: not (x>7854 and (x-7855)%4==0),delim_whitespace=True,names=['name','A','Z','N','spin','Mass excess (Mev)','source'])
def index_winv_v2(z,n):
        
    idx_start = np.searchsorted(nuclear_data['Z'], z, side='left')
    idx_end = np.searchsorted(nuclear_data['Z'], z, side='right')
    if idx_end>idx_start:
        j=np.searchsorted(nuclear_data['N'][idx_start:idx_end],n)+idx_start
        if nuclear_data['N'][j]==n and nuclear_data['Z'][j]==z:
            return j
        else:
            return 'None'
    else:
        return 'None'

Rates used in the simulation

In [ ]:
rates_Reaclib_winnet=pyna.rates.library.Library(
    libfile=r"Nuclear_data\decays\actual\Reaclib_18_9_20_sin_teoricas_beta_alpha_reaclib1"
    )

# Remove duplicate links from the library
rates_to_remove = []
for pair in rates_Reaclib_winnet.find_duplicate_links():
    for r in pair:
        if isinstance(r, pyna.rates.ReacLibRate):
            rates_to_remove.append(r)
for r in rates_to_remove:
    rates_Reaclib_winnet.remove_rate(r)

####Checking
len(rates_Reaclib_winnet.find_duplicate_links())


filter_alpha=pyna.RateFilter(
    products=['he4'],
    exact=False,
    max_reactants=1,
    max_products=2,
    filter_function=lambda r: r.Q>0 and r.reactants[0].Z==r.products[0].Z+r.products[1].Z)

rates_alpha=rates_Reaclib_winnet.filter(filter_alpha)

filter_beta_minus=pyna.RateFilter(
    max_reactants=1,
    max_products=1,
    filter_function=lambda r: r.Q>0 and r.reactants[0]==r.products[0]+1)

rates_beta_minus=rates_Reaclib_winnet.filter(filter_beta_minus)

print('number of alpha decays '+str(len(rates_alpha.get_rates()))+', number of beta decays '+str(len(rates_beta_minus.get_rates())))


Calcualtion of $\epsilon (t)$

In [ ]:

e_b=np.zeros(N_steps)
e_a=np.zeros(N_steps)
e_b_nuclei=[]
e_a_nuclei=[]
nuclei_b=[]
nuclei_a=[]
Xa=np.zeros(N_steps)

y_alpha=np.zeros(len(time_Winnet))
n_alpha=np.zeros(len(time_Winnet))

for beta_rate in tqdm(rates_beta_minus.get_rates()):
    nuclei=beta_rate.reactants[0]
    decay_rate_i=beta_rate.eval(1e7)
    Q_i=beta_rate.Q
    Z_i=nuclei.Z
    N_i=nuclei.N
    m_i=nuclear_data['Mass excess (Mev)'][index_winv_v2(Z_i,N_i)]+(Z_i+N_i)*m_u
    X_i=abundance_time_of_nuclei_n_z(N_i,Z_i)
    e=decay_rate_i*(Q_i*X_i*c**2)/m_i
    e_b+=e
    e_b_nuclei.append(e)
    nuclei_b.append(nuclei)
    

for alpha_rate in tqdm(rates_alpha.get_rates()):
    nuclei=alpha_rate.reactants[0]
    decay_rate_i=alpha_rate.eval(1e7)
    Q_i=alpha_rate.Q
    Z_i=nuclei.Z
    N_i=nuclei.N
    m_i=nuclear_data['Mass excess (Mev)'][index_winv_v2(Z_i,N_i)]+(Z_i+N_i)*m_u
    X_i=abundance_time_of_nuclei_n_z(N_i,Z_i)
    
    e=decay_rate_i*(Q_i*X_i*c**2)/m_i
    e_a+=e
    e_a_nuclei.append(e)
    nuclei_a.append(nuclei)
    if Z_i+N_i>=210:
        Xa+=X_i

    Y_i=abundances_winnet[:, index_out_win_n_z(N_i,Z_i), 2]
    y_alpha+=(decay_rate_i)*Y_i
    n_alpha+=(decay_rate_i)*Y_i*np.exp(-decay_rate_i*time_Winnet)


e_a_erg=e_a*1e4
e_b_erg=e_b*1e4
e_a_nuclei_erg=np.array(e_a_nuclei)*1e4
e_b_nuclei_erg=np.array(e_b_nuclei)*1e4
e_a_nuclei_frac=e_a_nuclei/e_a
e_b_nuclei_frac=e_b_nuclei/e_b

In [ ]:
'''find relevant nuclei'''
time_days=time*8.64e-4
i_initial=np.searchsorted(time_days, 0.1)
i_Final=100000

i_s=np.argsort(-np.max(e_b_nuclei_frac[:,i_initial:i_Final], axis=1))
e_b_nuclei_sorted=[e_b_nuclei_erg[i] for i in i_s]
e_b_nuclei_frac_sorted=[e_b_nuclei_frac[i] for i in i_s]
nuclei_b_sorted=[nuclei_b[i] for i in i_s]

i_s=np.argsort(-np.max(e_a_nuclei_frac[:,i_initial:i_Final], axis=1))
e_a_nuclei_sorted=[e_a_nuclei_erg[i] for i in i_s]
e_a_nuclei_frac_sorted=[e_a_nuclei_frac[i] for i in i_s]    
nuclei_a_sorted=[nuclei_a[i] for i in i_s]

In [ ]:
pd.DataFrame({'Epsilon_alpha':e_a_erg,'Time [Days]':time_days}).to_csv('Results/actual/epsilon_alpha.csv',index=False)
pd.DataFrame({'Epsilon_beta':e_b_erg,'Time [Days]':time_days}).to_csv('Reulsts/actual/epsilon_beta.csv',index=False)

In [ ]:

for i in range(20):
    pd.DataFrame({'time_days':time_days,f'{nuclei_b_sorted[i]} top {i+1}':e_b_nuclei_sorted[i]}).to_csv(f'Results/actual/beta/e_r_{i}.csv',index=False)
for i in range(20):
    pd.DataFrame({'time_days':time_days,f'{nuclei_a_sorted[i]} top {i+1}':e_a_nuclei_sorted[i]}).to_csv(f'Results/actual/alpha/e_r_{i}.csv',index=False)